# Lesson 5 — Hamiltonian Simulation and Trotterization

**Quantum Computing with Qiskit II**

This notebook accompanies Lesson 5. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- Why computing $e^{-iHt}$ classically is exponentially hard
- Pauli decomposition and `SparsePauliOp`
- The transverse-field Ising model as a running example
- First-order Lie-Trotter product formula and circuit primitives
- Second-order Suzuki-Trotter and infidelity scaling
- Simulation error vs circuit depth on a log-log plot
- `PauliEvolutionGate` with `LieTrotter` and `SuzukiTrotter` synthesizers

In [ ]:
!pip install qiskit qiskit-aer scipy pylatexenc --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

import qiskit
import qiskit_aer

print("Qiskit:    ", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)

In [ ]:
# Core imports used throughout the notebook
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

print("Imports complete.")

## 1. The Transverse-Field Ising Model

The **transverse-field Ising model (TFIM)** is the running example for this lesson. For a one-dimensional chain of $n$ qubits:

$$H = J \sum_{i=0}^{n-2} Z_i Z_{i+1} + g \sum_{i=0}^{n-1} X_i$$

- The $ZZ$ terms couple neighboring qubits.
- The $X$ terms (transverse field) drive quantum fluctuations, mixing $|0\rangle$ and $|1\rangle$.

In Qiskit's `SparsePauliOp` string convention, the **rightmost character corresponds to qubit 0**. For $n = 4$, the Pauli strings are:

| Operator | String |
|---|---|
| $Z_0 Z_1$ | `"IIZZ"` |
| $Z_1 Z_2$ | `"IZZI"` |
| $Z_2 Z_3$ | `"ZZII"` |
| $X_0$ | `"IIIX"` |
| $X_1$ | `"IIXI"` |
| $X_2$ | `"IXII"` |
| $X_3$ | `"XIII"` |

The character at string position $n - 1 - k$ corresponds to qubit $k$.

In [ ]:
# Build the 4-qubit TFIM Hamiltonian as a SparsePauliOp
n = 4
J = 1.0    # ZZ coupling strength
g = 0.5    # transverse field strength

pauli_terms, coeffs = [], []

# ZZ terms: Z_i Z_{i+1} for i = 0, 1, ..., n-2
for i in range(n - 1):
    s = ['I'] * n
    s[n - 1 - i]       = 'Z'   # qubit i
    s[n - 1 - (i + 1)] = 'Z'   # qubit i+1
    pauli_terms.append(''.join(s))
    coeffs.append(J)

# X terms: X_i for i = 0, 1, ..., n-1
for i in range(n):
    s = ['I'] * n
    s[n - 1 - i] = 'X'
    pauli_terms.append(''.join(s))
    coeffs.append(g)

H_op = SparsePauliOp(pauli_terms, coeffs=coeffs)

print("TFIM Hamiltonian (n=4, J=1.0, g=0.5):")
print(f"  Number of Pauli terms: {len(H_op)}")
print()
for term, coeff in zip(pauli_terms, coeffs):
    print(f"  {coeff:+.1f} * {term}")
# Expected: 7 terms total.
# ZZ terms: IIZZ, IZZI, ZZII  (each with coefficient +1.0)
# X  terms: IIIX, IIXI, IXII, XIII  (each with coefficient +0.5)

In [ ]:
# Inspect the Hamiltonian matrix
H_matrix = H_op.to_matrix()         # 16x16 complex Hermitian matrix
eigenvalues = np.linalg.eigvalsh(H_matrix)   # sorted real eigenvalues

print(f"Matrix size:                      {H_matrix.shape}")
print(f"Is Hermitian:                     {np.allclose(H_matrix, H_matrix.conj().T)}")
print(f"Ground state energy (lowest ev):  {eigenvalues[0]:.4f}")
print(f"Highest eigenvalue:               {eigenvalues[-1]:.4f}")
print(f"Spectral width (max - min):       {eigenvalues[-1] - eigenvalues[0]:.4f}")
# Expected: 16x16, Hermitian=True.
# Lowest eigenvalue ≈ -3.43, highest ≈ +3.43 (symmetric spectrum for this parameter choice).
# The spectral width ≈ 6.85 sets the characteristic time scale of the dynamics.

## 2. Pauli Decomposition

Any $n$-qubit Hermitian operator $H$ can be written as:

$$H = \sum_{\mathbf{P} \in \{I,X,Y,Z\}^n} c_{\mathbf{P}} \; P_0 \otimes P_1 \otimes \cdots \otimes P_{n-1}$$

The coefficients are extracted by the **trace inner product**:

$$c_{\mathbf{P}} = \frac{1}{2^n} \operatorname{Tr}(\mathbf{P} \cdot H)$$

The cell below demonstrates this on a small 2-qubit example and verifies that `SparsePauliOp` recovers the same coefficients.

In [ ]:
# Manual Pauli decomposition of the 2-qubit TFIM (J=1, g=0.5)
# H_2q = 1.0 * Z0Z1 + 0.5 * X0 + 0.5 * X1
#
# np.kron(A, B) = A tensor B where A acts on the higher-index qubit.
# This is consistent with Qiskit's matrix convention (qubit 0 = rightmost = least significant).

I2 = np.eye(2)
X2 = np.array([[0, 1], [1, 0]])
Y2 = np.array([[0, -1j], [1j, 0]])
Z2 = np.array([[1, 0], [0, -1]])

# Build all 16 two-qubit Pauli strings
single_paulis = {'I': I2, 'X': X2, 'Y': Y2, 'Z': Z2}
paulis_2q = {p1 + p0: np.kron(M1, M0)
             for p1, M1 in single_paulis.items()
             for p0, M0 in single_paulis.items()}
# Note: key 'p1p0' -> kron(M1, M0) where p1 is qubit 1, p0 is qubit 0.
# Matches Qiskit's string convention: leftmost character = highest qubit index.

# Build the 2-qubit TFIM matrix
H_2q = 1.0 * np.kron(Z2, Z2) + 0.5 * np.kron(I2, X2) + 0.5 * np.kron(X2, I2)

# Extract coefficients via the trace formula: c_P = Tr(P @ H) / 2^n
print("Pauli decomposition of 2-qubit TFIM (nonzero terms):")
print(f"  {'String':>6}  {'Coefficient':>12}")
print("-" * 26)
for name, P in sorted(paulis_2q.items()):
    coeff = np.trace(P @ H_2q).real / 4
    if abs(coeff) > 1e-10:
        print(f"  {name:>6}  {coeff:>12.4f}")
print()

# Cross-check with SparsePauliOp
H_2q_op = SparsePauliOp(["ZZ", "IX", "XI"], coeffs=[1.0, 0.5, 0.5])
print("Cross-check: SparsePauliOp matrix matches H_2q:",
      np.allclose(H_2q_op.to_matrix(), H_2q))
# Expected nonzero terms: ZZ -> 1.0000, IX -> 0.5000, XI -> 0.5000
# Cross-check should print True

## 3. Circuit Primitives for Pauli Exponentials

Trotterization decomposes $e^{-iHt}$ into products of simpler exponentials. The key circuit identities are:

- $e^{-i\theta X_k} = R_x(2\theta, k)$
- $e^{-i\theta Z_k} = R_z(2\theta, k)$
- $e^{-i\theta Z_i Z_j} = \mathrm{CX}(i,j)\; R_z(2\theta, j)\; \mathrm{CX}(i,j)$

The ZZ identity holds because $\mathrm{CX}(i,j)$ maps the $Z_i Z_j$ parity information onto qubit $j$ alone, where a single $R_z$ can apply the correct phase.

In [ ]:
def zz_evolution(qc, i, j, theta):
    """Apply e^{-i theta Z_i Z_j} to circuit qc."""
    qc.cx(i, j)
    qc.rz(2 * theta, j)
    qc.cx(i, j)


# Verify the ZZ circuit identity on the state |00>
# e^{-i(pi/4) Z0Z1}|00> should give e^{-i*pi/4}|00> (phase of -i*pi/4 since Z0Z1|00>=+|00>)
theta_test = np.pi / 4
qc_zz = QuantumCircuit(2)
zz_evolution(qc_zz, 0, 1, theta_test)

sv_zz   = Statevector(qc_zz).data
expected = np.exp(-1j * theta_test) * np.array([1, 0, 0, 0])
print("Verify e^{-i(pi/4) Z0Z1} on |00>:")
print(f"  Circuit output:  {sv_zz}")
print(f"  Expected:        {expected}")
print(f"  Match:           {np.allclose(sv_zz, expected)}")
print()

# Verify on |11> (eigenvalue +1 as well)
qc_zz_11 = QuantumCircuit(2)
qc_zz_11.x([0, 1])   # prepare |11>
zz_evolution(qc_zz_11, 0, 1, theta_test)
sv_zz_11   = Statevector(qc_zz_11).data
expected_11 = np.exp(-1j * theta_test) * np.array([0, 0, 0, 1])
print("Verify e^{-i(pi/4) Z0Z1} on |11>:")
print(f"  Match: {np.allclose(sv_zz_11, expected_11)}")
print()

# Verify on |01> (eigenvalue -1, phase should be e^{+i*pi/4})
qc_zz_01 = QuantumCircuit(2)
qc_zz_01.x(0)   # prepare |01> (qubit 0 in |1>)
zz_evolution(qc_zz_01, 0, 1, theta_test)
sv_zz_01   = Statevector(qc_zz_01).data
expected_01 = np.exp(+1j * theta_test) * np.array([0, 1, 0, 0])
print("Verify e^{-i(pi/4) Z0Z1} on |01> (Z0Z1 eigenvalue -1):")
print(f"  Match: {np.allclose(sv_zz_01, expected_01)}")
# All three should print True

In [ ]:
# Draw the ZZ evolution circuit
qc_zz_draw = QuantumCircuit(2, name="ZZ")
zz_evolution(qc_zz_draw, 0, 1, theta_test)
qc_zz_draw.draw('mpl')

## 4. First-Order Lie-Trotter Circuit

The first-order Trotter formula approximates the time-evolution operator for $H = H_{ZZ} + H_X$ as:

$$e^{-iHt} \approx \left( e^{-iH_{ZZ}\,\Delta t} \cdot e^{-iH_X\,\Delta t} \right)^r \qquad \Delta t = \frac{t}{r}$$

Each factor is exact: all $ZZ$ terms commute with each other (so their product of exponentials equals the exponential of their sum), and likewise for all $X$ terms. The only approximation is the splitting between $H_{ZZ}$ and $H_X$.

In [ ]:
def trotter_step_1st(n, J, g, dt):
    """One first-order Lie-Trotter step for the TFIM.
    Applies e^{-i H_ZZ dt} then e^{-i H_X dt}."""
    qc = QuantumCircuit(n)
    # ZZ bonds: e^{-i J dt Z_i Z_{i+1}}
    for i in range(n - 1):
        zz_evolution(qc, i, i + 1, J * dt)
    # X field: e^{-i g dt X_i} = Rx(2 g dt, i)
    for i in range(n):
        qc.rx(2 * g * dt, i)
    return qc


def trotter_circuit_1st(n, J, g, t, r):
    """Full r-step first-order Trotter circuit for total evolution time t."""
    dt   = t / r
    qc   = QuantumCircuit(n)
    step = trotter_step_1st(n, J, g, dt)
    for _ in range(r):
        qc = qc.compose(step)
    return qc


# Inspect gate counts for a single step
step_example = trotter_step_1st(n, J, g, dt=0.25)
print("Gate counts in one first-order Trotter step (dt=0.25):")
print(f"  {dict(step_example.count_ops())}")
print(f"  Circuit depth: {step_example.depth()}")
# Expected: cx: 6 (2 CX per ZZ bond, 3 bonds), rz: 3 (1 per bond), rx: 4 (1 per site)
# Total two-qubit gates: 6 CX

In [ ]:
# Draw one Trotter step (dt = t/r with t=1.0, r=4)
trotter_step_1st(n, J, g, dt=0.25).draw('mpl')

## 5. Comparing with Exact Time Evolution

The exact time-evolved state is $|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle$. For a small system we can compute the matrix exponential directly with `scipy.linalg.expm` and compare it to the Trotter approximation.

The **infidelity** $1 - |\langle\psi_\mathrm{exact}|\psi_\mathrm{Trotter}\rangle|^2$ measures how far the Trotter state is from the exact state. It equals zero for perfect agreement and one for orthogonal states.

In [ ]:
# Initial state: |0000> (the default state for all qubits initialized to |0>)
psi0 = np.zeros(2**n, dtype=complex)
psi0[0] = 1.0

def exact_state(t):
    """Return the exact time-evolved state |psi(t)> = e^{-iHt} |0000>."""
    U = expm(-1j * H_matrix * t)
    return U @ psi0

def infidelity(psi1, psi2):
    """Compute 1 - |<psi1|psi2>|^2."""
    return 1.0 - abs(np.dot(psi1.conj(), psi2))**2


# Sanity check: at t=0 the exact and Trotter states are both |0000>
psi_exact_0 = exact_state(0.0)
qc_r1       = trotter_circuit_1st(n, J, g, t=0.0, r=1)
print(f"Infidelity at t=0: {infidelity(psi_exact_0, Statevector(qc_r1).data):.2e}")
print(f"  (should be ~0 -- both are |0000>)")
print()

# Compare exact vs Trotter at t=1.0 for several values of r
t_sim = 1.0
psi_exact = exact_state(t_sim)

print(f"{'r':>4}  {'Infidelity':>14}  {'Circuit depth':>14}")
print("-" * 38)
for r in [1, 2, 4, 8, 16]:
    qc  = trotter_circuit_1st(n, J, g, t_sim, r)
    err = infidelity(psi_exact, Statevector(qc).data)
    print(f"  {r:2d}  {err:14.4e}  {qc.depth():14d}")
# Expected (approximate values for n=4, J=1.0, g=0.5, t=1.0):
#   r= 1:  6.71e-01  depth=7    (large error -- one coarse step)
#   r= 2:  1.31e-01  depth=14
#   r= 4:  2.94e-02  depth=28
#   r= 8:  7.06e-03  depth=56
#   r=16:  1.73e-03  depth=112
# Infidelity decreases by roughly a factor of 4 each time r doubles (slope -2 on log-log).

## 6. Second-Order Suzuki-Trotter

The **second-order Suzuki-Trotter (symmetric) step** achieves lower error for the same number of steps by applying the $ZZ$ half-step on both sides of the $X$ full step:

$$e^{-iH\Delta t} \approx e^{-iH_{ZZ}\frac{\Delta t}{2}} \cdot e^{-iH_X \Delta t} \cdot e^{-iH_{ZZ}\frac{\Delta t}{2}}$$

The leading error term from BCH, proportional to $[H_{ZZ}, [H_{ZZ}, H_X]] - [H_X, [H_X, H_{ZZ}]]$, is antisymmetric and cancels between the forward and backward half-steps. The error per step drops from $O(\Delta t^2)$ to $O(\Delta t^3)$, giving total infidelity $O(1/r^4)$ instead of $O(1/r^2)$.

In [ ]:
def trotter_step_2nd(n, J, g, dt):
    """One second-order symmetric Trotter step for the TFIM.
    Applies e^{-i H_ZZ dt/2} then e^{-i H_X dt} then e^{-i H_ZZ dt/2}."""
    qc = QuantumCircuit(n)
    # ZZ forward half-step
    for i in range(n - 1):
        zz_evolution(qc, i, i + 1, J * dt / 2)
    # X full step
    for i in range(n):
        qc.rx(2 * g * dt, i)
    # ZZ backward half-step
    for i in range(n - 1):
        zz_evolution(qc, i, i + 1, J * dt / 2)
    return qc


def trotter_circuit_2nd(n, J, g, t, r):
    """Full r-step second-order Trotter circuit for total evolution time t."""
    dt   = t / r
    qc   = QuantumCircuit(n)
    step = trotter_step_2nd(n, J, g, dt)
    for _ in range(r):
        qc = qc.compose(step)
    return qc


# Gate count comparison: one step
step_1st = trotter_step_1st(n, J, g, dt=0.25)
step_2nd = trotter_step_2nd(n, J, g, dt=0.25)
print("Gate counts per Trotter step (dt=0.25):")
print(f"  1st order: {dict(step_1st.count_ops())}  depth={step_1st.depth()}")
print(f"  2nd order: {dict(step_2nd.count_ops())}  depth={step_2nd.depth()}")
# Expected: 2nd order has twice as many CX and Rz gates as 1st order.
# 1st order: cx=6, rz=3, rx=4
# 2nd order: cx=12, rz=6, rx=4  (the extra 6 CX+3 Rz are the second ZZ half-step)

In [ ]:
# Compare infidelities: first-order vs second-order for the same r values
r_values = [1, 2, 4, 8, 16, 32]

infidelities_1st, infidelities_2nd = [], []
for r in r_values:
    psi_1st = Statevector(trotter_circuit_1st(n, J, g, t_sim, r)).data
    psi_2nd = Statevector(trotter_circuit_2nd(n, J, g, t_sim, r)).data
    infidelities_1st.append(infidelity(psi_exact, psi_1st))
    infidelities_2nd.append(infidelity(psi_exact, psi_2nd))

print(f"{'r':>4}  {'1st-order infidelity':>22}  {'2nd-order infidelity':>22}")
print("-" * 54)
for r, e1, e2 in zip(r_values, infidelities_1st, infidelities_2nd):
    print(f"  {r:2d}  {e1:22.4e}  {e2:22.4e}")
# Expected (approximate values for n=4, J=1.0, g=0.5, t=1.0):
#   r= 1:  6.71e-01  vs  2.18e-01
#   r= 2:  1.31e-01  vs  6.32e-03
#   r= 4:  2.94e-02  vs  3.36e-04   (~90x better at same r)
#   r= 8:  7.06e-03  vs  2.02e-05   (~350x better)
#   r=16:  1.73e-03  vs  1.25e-06   (~1400x better)
#   r=32:  4.30e-04  vs  7.81e-08
# 2nd-order infidelity decreases by ~16x per doubling of r (slope -4 on log-log).

In [ ]:
# Log-log plot: infidelity vs number of Trotter steps
r_arr = np.array(r_values, dtype=float)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(r_arr, infidelities_1st, 'o-', color='tomato',    linewidth=2, markersize=7,
          label='1st-order Trotter')
ax.loglog(r_arr, infidelities_2nd, 's-', color='steelblue', linewidth=2, markersize=7,
          label='2nd-order Suzuki')

# Reference slope lines
r_ref = np.array([1.0, 32.0])
ref_1st = infidelities_1st[0] * (r_ref / r_values[0])**(-2)
ref_2nd = infidelities_2nd[0] * (r_ref / r_values[0])**(-4)
ax.loglog(r_ref, ref_1st, ':', color='tomato',    alpha=0.5, linewidth=1.5, label='slope $-2$')
ax.loglog(r_ref, ref_2nd, ':', color='steelblue', alpha=0.5, linewidth=1.5, label='slope $-4$')

ax.set_xlabel("Number of Trotter steps $r$", fontsize=12)
ax.set_ylabel("Infidelity", fontsize=12)
ax.set_title("Trotter error scaling (TFIM, $n=4$, $J=1$, $g=0.5$, $t=1$)", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
# Expected: both curves are approximately straight lines on the log-log axes.
# 1st-order slope ≈ -2 (infidelity ∝ 1/r^2).
# 2nd-order slope ≈ -4 (infidelity ∝ 1/r^4).
# The reference slope lines should closely track the data.

## 7. Time Evolution of a Physical Observable

The infidelity measures the total error between states. A complementary view is to track a physical observable over time and compare the exact and Trotter curves.

The local magnetization $\langle Z_0 \rangle(t)$ starts at $+1$ (since $|0000\rangle$ is a $Z$ eigenstate) and oscillates as the transverse field drives the system away from the initial state.

In [ ]:
# Track <Z_0>(t) under the TFIM dynamics
Z0_op     = SparsePauliOp("IIIZ")        # Z on qubit 0 (4-qubit system; rightmost = qubit 0)
Z0_matrix = Z0_op.to_matrix()            # 16x16 matrix

def expectation_Z0(psi):
    """Compute <psi|Z_0|psi>."""
    return (psi.conj() @ Z0_matrix @ psi).real


t_values = np.linspace(0.0, 2.0, 60)

Z0_exact     = [expectation_Z0(exact_state(t)) for t in t_values]

# Trotter curves for r=2 (coarse) and r=8 (fine)
Z0_trotter_2 = [expectation_Z0(Statevector(trotter_circuit_1st(n, J, g, t, r=2)).data)
                for t in t_values]
Z0_trotter_8 = [expectation_Z0(Statevector(trotter_circuit_1st(n, J, g, t, r=8)).data)
                for t in t_values]

plt.figure(figsize=(8, 4))
plt.plot(t_values, Z0_exact,     color='black',     linewidth=2,   label='Exact')
plt.plot(t_values, Z0_trotter_2, color='tomato',    linewidth=1.5, linestyle='--',
         label='Trotter 1st order, $r=2$')
plt.plot(t_values, Z0_trotter_8, color='steelblue', linewidth=1.5, linestyle=':',
         label='Trotter 1st order, $r=8$')

plt.xlabel("Time $t$", fontsize=12)
plt.ylabel(r"$\langle Z_0 \rangle$", fontsize=12)
plt.title(r"Time evolution of $\langle Z_0\rangle$ under TFIM ($n=4$, $J=1$, $g=0.5$)",
          fontsize=12)
plt.legend(fontsize=10)
plt.ylim(-1.1, 1.1)
plt.axhline(0, color='gray', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()
# Expected: <Z_0> starts at +1 and oscillates as the transverse field mixes |0> and |1>.
# r=8 Trotter closely tracks the exact curve for t up to ~2.
# r=2 Trotter shows visible deviations especially at larger t.

## 8. PauliEvolutionGate in Qiskit

`PauliEvolutionGate` is Qiskit's high-level gate for the time-evolution operator $e^{-iHt}$. It takes a `SparsePauliOp` and a synthesis strategy (`LieTrotter`, `SuzukiTrotter`, etc.) and automatically decomposes into primitive gates when `.decompose()` is called.

The `reps` parameter sets how many Trotter repetitions are used. Each repetition simulates time $t / \text{reps}$, so the total simulated time is always $t$.

In [ ]:
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis.evolution import LieTrotter, SuzukiTrotter

r_demo = 4

# First-order Lie-Trotter
gate_lie = PauliEvolutionGate(H_op, time=t_sim, synthesis=LieTrotter(reps=r_demo))
qc_lie   = QuantumCircuit(n)
qc_lie.append(gate_lie, range(n))
qc_lie_decomp = qc_lie.decompose()

print(f"LieTrotter (reps={r_demo}) circuit:")
print(f"  Gate counts: {dict(qc_lie_decomp.count_ops())}")
print(f"  Depth:       {qc_lie_decomp.depth()}")

# Verify the statevector matches the manual first-order circuit
psi_lie = Statevector(qc_lie_decomp).data
psi_man = Statevector(trotter_circuit_1st(n, J, g, t_sim, r_demo)).data
print(f"  Infidelity vs manual 1st-order circuit: {infidelity(psi_lie, psi_man):.2e}")
# Expected: infidelity ≈ 0 (both circuits implement the same LieTrotter formula).
# Gate counts: cx=24, rz=12, rx=16 for r=4 steps (matches 4 * {cx:6, rz:3, rx:4})

In [ ]:
# Second-order Suzuki-Trotter
gate_suz = PauliEvolutionGate(H_op, time=t_sim, synthesis=SuzukiTrotter(order=2, reps=r_demo))
qc_suz   = QuantumCircuit(n)
qc_suz.append(gate_suz, range(n))
qc_suz_decomp = qc_suz.decompose()

print(f"SuzukiTrotter order=2 (reps={r_demo}) circuit:")
print(f"  Gate counts: {dict(qc_suz_decomp.count_ops())}")
print(f"  Depth:       {qc_suz_decomp.depth()}")

# Verify against the manual second-order circuit
psi_suz  = Statevector(qc_suz_decomp).data
psi_man2 = Statevector(trotter_circuit_2nd(n, J, g, t_sim, r_demo)).data
print(f"  Infidelity vs manual 2nd-order circuit: {infidelity(psi_suz, psi_man2):.2e}")
# Expected: infidelity ≈ 0 (both implement the same second-order Suzuki formula).
# Gate counts approximately double compared to LieTrotter: cx≈48, rz≈24, rx=16 for r=4.

In [ ]:
# Circuit depth vs infidelity: compare both formulas at the same r values
print(f"{'r':>4}  {'1st depth':>10}  {'2nd depth':>10}  {'1st infidelity':>16}  {'2nd infidelity':>16}")
print("-" * 62)
for r in [1, 2, 4, 8, 16]:
    qc1 = trotter_circuit_1st(n, J, g, t_sim, r)
    qc2 = trotter_circuit_2nd(n, J, g, t_sim, r)
    e1  = infidelity(psi_exact, Statevector(qc1).data)
    e2  = infidelity(psi_exact, Statevector(qc2).data)
    print(f"  {r:2d}  {qc1.depth():10d}  {qc2.depth():10d}  {e1:16.4e}  {e2:16.4e}")
# Expected: depth grows linearly with r.
# For similar depths (e.g. 1st r=16 vs 2nd r=8), the 2nd-order circuit has much lower infidelity.
# This demonstrates the advantage of higher-order formulas on depth-limited hardware.

## 9. Exercises

Work through each exercise in the cell below it. Expected results are provided so you can check your work.

### Exercise 1: Verify the ZZ rotation identity using matrix exponentiation

Build the circuit for $e^{-i(\pi/8)\, Z_0 Z_1}$ using `zz_evolution` and verify that the resulting `Statevector` matches the direct matrix exponential $\exp(-i(\pi/8)\cdot Z\otimes Z)$ applied to several basis states.

1. Use `scipy.linalg.expm` to compute the $4\times4$ exact unitary $U = e^{-i(\pi/8)\, Z\otimes Z}$.
2. For each of the four basis states $|00\rangle, |01\rangle, |10\rangle, |11\rangle$, apply $U$ directly and compare to the circuit output.
3. Report whether all four states match.

*Expected: all four states match to numerical precision (infidelity $< 10^{-14}$).*

In [ ]:
# Exercise 1: Verify ZZ rotation circuit against matrix exponentiation

theta_ex1 = np.pi / 8
ZZ_matrix = np.kron(Z2, Z2)   # 4x4 matrix for Z1 tensor Z0

# TODO: compute U_exact = expm(-1j * theta_ex1 * ZZ_matrix)
# U_exact = ...

# TODO: for each basis state |00>, |01>, |10>, |11>, apply the circuit and compare
# basis_states = [np.array([1,0,0,0]), np.array([0,1,0,0]),
#                 np.array([0,0,1,0]), np.array([0,0,0,1])]
# basis_labels = ['|00>', '|01>', '|10>', '|11>']
#
# for label, b in zip(basis_labels, basis_states):
#     # Build circuit starting from this basis state
#     qc_ex1 = QuantumCircuit(2)
#     qc_ex1.initialize(b, [0, 1])   # initialize both qubits from the 4-element vector
#     zz_evolution(qc_ex1, 0, 1, theta_ex1)
#     psi_circuit = Statevector(qc_ex1).data
#     psi_exact_b = U_exact @ b
#     err = infidelity(psi_exact_b, psi_circuit)
#     print(f"  {label}: infidelity = {err:.2e}")

### Exercise 2: Add a Heisenberg YY coupling

Extend the TFIM to the **XXZ model** by adding a $YY$ coupling:

$$H_{XXZ} = J\sum_i (Z_i Z_{i+1} + Y_i Y_{i+1}) + g\sum_i X_i$$

1. Build the `SparsePauliOp` for $H_{XXZ}$ with $n=4$, $J=1.0$, $g=0.5$.
2. Implement a `yy_evolution(qc, i, j, theta)` function that applies $e^{-i\theta Y_i Y_j}$.
   Hint: $e^{-i\theta Y_i Y_j} = (S^\dagger H)_i (S^\dagger H)_j \cdot e^{-i\theta Z_i Z_j} \cdot (H S)_i (H S)_j$ where $S^\dagger = R_z(-\pi/2) \cdot H \cdot R_z(\pi/4)$ ... or more simply, the $YY$ rotation can be obtained from the $ZZ$ rotation by conjugating each qubit with $R_x(\pi/2)$.
3. Build a first-order Trotter step for $H_{XXZ}$ and compare the infidelity curve vs $r$ to the plain TFIM.

*Expected: the $YY$ terms increase the error magnitude (larger commutator norm) but the $O(1/r^2)$ infidelity scaling persists.*

In [ ]:
# Exercise 2: Heisenberg YY coupling

# TODO: build SparsePauliOp for H_XXZ (add YY terms to the existing TFIM terms)
# For Y_i Y_{i+1} on a 4-qubit system:
#   i=0: IIYY, i=1: IYYI, i=2: YYII
# H_xxz_op = SparsePauliOp([...], coeffs=[...])

# TODO: implement yy_evolution
# Hint: Rx(pi/2) Z Rx(-pi/2) = -Y for each qubit independently.
# Applying Rx(pi/2) to both qubits maps ZiZj to (-Yi)(-Yj) = YiYj.
# The two sign flips cancel, so the circuit correctly implements e^{-i theta Y_i Y_j}.
#
# def yy_evolution(qc, i, j, theta):
#     qc.rx(np.pi / 2, i)         # rotate qubit i: Z -> -Y
#     qc.rx(np.pi / 2, j)         # rotate qubit j: Z -> -Y
#     zz_evolution(qc, i, j, theta)   # implement e^{-i theta (-Yi)(-Yj)} = e^{-i theta YiYj}
#     qc.rx(-np.pi / 2, i)        # undo rotation
#     qc.rx(-np.pi / 2, j)

# TODO: build the Trotter step for H_XXZ, compute infidelity vs r,
# and compare to the TFIM infidelity curve from cell-19

### Exercise 3: Minimum $r$ for a target infidelity

For the TFIM with $J=1.0$, $g=0.5$, $t=1.0$:

1. Find the smallest $r$ such that the first-order Trotter circuit achieves infidelity $< 10^{-4}$.
2. Find the smallest $r$ such that the second-order Trotter circuit achieves the same target.
3. Compute the total CX gate count for each circuit at its minimum $r$ and compare.

*Expected: the second-order circuit needs far fewer steps and therefore fewer gates to meet the target. The CX gate ratio should be close to $(r_1 / r_2)$ since each step has roughly twice the CX count.*

In [ ]:
# Exercise 3: Minimum r for infidelity < 1e-4

target = 1e-4

# TODO: loop over increasing r values and find the minimum r for each Trotter order
#
# r_min_1st, r_min_2nd = None, None
# for r in range(1, 100):
#     if r_min_1st is None:
#         e1 = infidelity(psi_exact, Statevector(trotter_circuit_1st(n, J, g, t_sim, r)).data)
#         if e1 < target:
#             r_min_1st = r
#     if r_min_2nd is None:
#         e2 = infidelity(psi_exact, Statevector(trotter_circuit_2nd(n, J, g, t_sim, r)).data)
#         if e2 < target:
#             r_min_2nd = r
#     if r_min_1st and r_min_2nd:
#         break
#
# cx_1st = trotter_circuit_1st(n, J, g, t_sim, r_min_1st).count_ops().get('cx', 0)
# cx_2nd = trotter_circuit_2nd(n, J, g, t_sim, r_min_2nd).count_ops().get('cx', 0)
# print(f"1st order: r={r_min_1st}, CX count={cx_1st}")
# print(f"2nd order: r={r_min_2nd}, CX count={cx_2nd}")
# print(f"CX reduction factor: {cx_1st / cx_2nd:.1f}x")

### Exercise 4: Fourth-order Suzuki-Trotter

Use `PauliEvolutionGate` with `SuzukiTrotter(order=4, reps=r)` to build fourth-order Trotter circuits and add their infidelity curve to the log-log plot from cell-20.

1. Compute the fourth-order infidelities for $r \in \{1, 2, 4, 8, 16\}$.
2. Add the curve to the log-log plot with a reference slope line of $-8$.
3. Count the total gates in the fourth-order circuit at $r=4$ and compare to the second-order circuit at the same $r$.

*Expected: fourth-order slope ≈ $-8$ on the log-log plot. The gate count per step is about 5 times larger than second order (from Suzuki's recursive construction), but the error drops much faster with $r$.*

In [ ]:
# Exercise 4: Fourth-order Suzuki-Trotter infidelity

# TODO: compute infidelities for order=4
# r_values_ex4 = [1, 2, 4, 8, 16]
# infidelities_4th = []
# for r in r_values_ex4:
#     gate_4th = PauliEvolutionGate(H_op, time=t_sim, synthesis=SuzukiTrotter(order=4, reps=r))
#     qc_4th   = QuantumCircuit(n)
#     qc_4th.append(gate_4th, range(n))
#     psi_4th  = Statevector(qc_4th.decompose()).data
#     infidelities_4th.append(infidelity(psi_exact, psi_4th))

# TODO: reproduce the log-log plot from cell-20 and add the 4th-order curve
# Add reference slope line: 0.5 * r_ref**(-8)

# TODO: compare gate counts at r=4 for order=2 and order=4
# gate_2nd_r4 = PauliEvolutionGate(H_op, time=t_sim, synthesis=SuzukiTrotter(order=2, reps=4))
# gate_4th_r4 = PauliEvolutionGate(H_op, time=t_sim, synthesis=SuzukiTrotter(order=4, reps=4))
# ... decompose and compare count_ops()

## Summary

In this lesson you:

- Built the transverse-field Ising Hamiltonian as a `SparsePauliOp` using the string convention (rightmost character = qubit 0) and verified the matrix representation against a manually computed Pauli decomposition via the trace formula $c_P = \text{Tr}(P \cdot H) / 2^n$.
- Derived and verified the three core circuit primitives: $e^{-i\theta X_k} = R_x(2\theta)$, $e^{-i\theta Z_k} = R_z(2\theta)$, and $e^{-i\theta Z_i Z_j} = \mathrm{CX}(i,j)\;R_z(2\theta,j)\;\mathrm{CX}(i,j)$.
- Implemented first-order (`trotter_circuit_1st`) and second-order (`trotter_circuit_2nd`) Trotter circuits and compared their statevectors against exact time evolution computed with `scipy.linalg.expm`.
- Produced a log-log error plot confirming infidelity $\propto 1/r^2$ for first order and $\propto 1/r^4$ for second order, and observed the depth-accuracy trade-off in the summary table.
- Reproduced both circuits using `PauliEvolutionGate` with `LieTrotter` and `SuzukiTrotter` synthesizers and verified agreement with the manual implementations.

**Lesson 6** builds on these tools to implement the Variational Quantum Eigensolver (VQE). The same `SparsePauliOp` decomposition reappears as the backbone of the energy measurement, and Trotter circuits provide physically motivated ansatz families for the variational optimization.